# 06. Bérleti Piac, Hozamszámítás és Rent Gap Elemzés

**Cél**: Az eladó és kiadó lakáspiaci szegmensek szisztematikus összehasonlítása, a bruttó és nettó bérleti hozamok (Gross / Net Rental Yield), a Price-to-Rent (P/R) ráta, valamint a Neil Smith-féle bérleti rés (Rent Gap) empirikus kimutatása Kőbányán.

**Módszertan és kettős hozamszámítás**:
- **1. Fajlagos (m²-alapú) bruttó hozam**: $\text{Gross Yield}_{\text{m}^2} = \frac{\text{Átlagos havi bérleti díj / m}^2 \times 12}{\text{Medián eladási ár / m}^2} \times 100\%$ (Kiküszöböli az eladó és kiadó minták méretbeli aszimmetriáját).
- **2. Egységár-alapú (lakásméretű) bruttó hozam**: $\text{Gross Yield}_{\text{egység}} = \frac{\text{Átlagos lakásbérleti díj} \times 12}{\text{Medián lakásár}} \times 100\%$ (A bérlemények átlagos mérete kisebb, ~48,7 m², szemben az eladók ~57,6 m²-es átlagával, ami ~0,6%-pontos strukturális eltérést eredményez).
- **Price-to-Rent (P/R)**: Megtérülési idő években (Vételár / Éves bérleti bevétel).
- **Rent Gap**: A jelenlegi állapotú hasznosítás tőkésített bérleti értéke és a felújítás után elérhető maximális potenciális bérleti érték közötti különbség.

In [ ]:
import warnings; warnings.filterwarnings('ignore')
import sys, os
from _utils import *
setup_plotly()
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import ipywidgets as widgets
from IPython.display import display, clear_output, HTML
import pandas as pd
import numpy as np

df = load_szamitott_master()
elado = df[df['listing_type'] == 'elado'].copy()
kiado = df[df['listing_type'] == 'kiado'].copy()
print(f"Adatbázis: {len(elado)} db eladó és {len(kiado)} db kiadó hirdetés.")

### 1. Bérleti Piac és Hozam KPI Mutatók
A kőbányai lakáskiadási piac alapvető megtérülési sarokszámai a kétféle módszertan szerint.

In [ ]:
atlag_berlet_huf = kiado['price_huf'].mean()
median_berlet_huf = kiado['price_huf'].median()
atlag_berlet_nm = kiado['nm_ar_huf'].mean()
median_elado_nm = elado['nm_ar_huf'].median()
median_elado_ar = elado['price_huf'].median()

# 1. Fajlagos m² alapú hozam
brutto_hozam_pct = (atlag_berlet_nm * 12 / median_elado_nm) * 100
pr_rata_ev = median_elado_nm / (atlag_berlet_nm * 12)

# 2. Egységár alapú hozam
brutto_hozam_egyseg_pct = (atlag_berlet_huf * 12 / median_elado_ar) * 100
pr_rata_egyseg_ev = median_elado_ar / (atlag_berlet_huf * 12)

kpi_cards = [
    ("Átlagos Havi Bérlet", fmt_huf(atlag_berlet_huf) + " / hó", f"Medián: {fmt_huf(median_berlet_huf)}", "#1e3a8a"),
    ("Bérleti Fajlagos Díj", fmt_huf(atlag_berlet_nm) + " / m²", "Havi fajlagos díj", "#2563eb"),
    ("Fajlagos Bruttó Hozam (m²)", f"{brutto_hozam_pct:.2f}%", f"P/R: {pr_rata_ev:.1f} év", "#059669"),
    ("Egységár Bruttó Hozam (lakás)", f"{brutto_hozam_egyseg_pct:.2f}%", f"P/R: {pr_rata_egyseg_ev:.1f} év", "#d97706"),
    ("Kiadó Lakások Aránya", f"{len(kiado)/len(df)*100:.1f}%", f"{len(kiado)} db hirdetés", "#7c3aed"),
    ("Nettó Hozam (85% kihaszn.)", f"{brutto_hozam_pct * 0.85 * 0.85:.2f}%", "Költségek levonása után", "#10b981")
]
display(HTML(kpi_grid_html(kpi_cards)))

### 2. Városrészi Bérleti Hozamok és Összehasonlítás
A lakásvásárlási és bérleti árak összevetése városrészenként.

In [ ]:
agg_elado = elado.groupby('varosresz')['nm_ar_huf'].median().reset_index(name='elado_nm_ar')
agg_kiado = kiado.groupby('varosresz')['nm_ar_huf'].median().reset_index(name='kiado_nm_ar')

merged_yield = pd.merge(agg_elado, agg_kiado, on='varosresz', how='inner')
merged_yield['brutto_hozam_pct'] = (merged_yield['kiado_nm_ar'] * 12 / merged_yield['elado_nm_ar']) * 100
merged_yield['pr_ratio'] = merged_yield['elado_nm_ar'] / (merged_yield['kiado_nm_ar'] * 12)

fig1 = make_subplots(specs=[[{"secondary_y": True}]])

fig1.add_trace(
    go.Bar(x=merged_yield['varosresz'], y=merged_yield['elado_nm_ar'], name='Eladási Ár/m² (HUF)', marker_color='#2563eb'),
    secondary_y=False
)
fig1.add_trace(
    go.Scatter(x=merged_yield['varosresz'], y=merged_yield['brutto_hozam_pct'], name='Bruttó Bérleti Hozam (%)', mode='lines+markers', line=dict(color='#10b981', width=3), marker=dict(size=10)),
    secondary_y=True
)

fig1.update_layout(
    title_text='Eladási négyzetméterárak és bruttó bérleti hozamok városrészenként',
    template=PLOTLY_TEMPLATE,
    height=450,
    xaxis_tickangle=-30
)
fig1.update_yaxes(title_text='Eladási Ár / m² (HUF)', secondary_y=False)
fig1.update_yaxes(title_text='Bruttó Hozam (%)', secondary_y=True)
fig1.show()

# Hozamtáblázat
display(HTML("<div style='max-width: 700px; margin: 15px 0;'>" + merged_yield.round(2).to_html(classes='table table-bordered table-striped', index=False) + "</div>"))

### 3. Smith-féle Rent Gap Elemzés (Bérleti Rés)
A felújított és felújítandó lakások közötti tőkésített bérleti különbség (Rent Gap) bemutatása.

In [ ]:
# Bérleti rés elemzése állapotonként
allapot_stat = df.groupby(['allapot', 'listing_type'])['nm_ar_huf'].median().unstack()
allapot_stat = allapot_stat.dropna()

if 'kiado' in allapot_stat.columns and 'elado' in allapot_stat.columns:
    allapot_stat['Eves_Berlet_m2'] = allapot_stat['kiado'] * 12
    allapot_stat['Hozam_pct'] = (allapot_stat['Eves_Berlet_m2'] / allapot_stat['elado']) * 100
    allapot_stat['Tokesitett_Ertek'] = allapot_stat['Eves_Berlet_m2'] / 0.05
    
    # Kiszámoljuk a felújítandó állapot és a legmagasabb (felújított/kiváló) állapot tőkésített értéke közötti különbséget (Rent Gap)
    max_potencial = allapot_stat['Tokesitett_Ertek'].max()
    allapot_stat['Rent_Gap'] = max_potencial - allapot_stat['Tokesitett_Ertek']
    
    fig2 = go.Figure()
    fig2.add_trace(go.Bar(
        x=allapot_stat.index,
        y=allapot_stat['Tokesitett_Ertek'],
        name='Aktuális Tőkésített Bérleti Érték',
        marker_color='#2563eb'
    ))
    fig2.add_trace(go.Bar(
        x=allapot_stat.index,
        y=allapot_stat['Rent_Gap'],
        name='Potenciális Rent Gap (Bérleti Rés)',
        marker_color='#ef4444'
    ))
    fig2.update_layout(
        barmode='stack',
        title='Neil Smith-féle Rent Gap (Bérleti Rés) Kőbányán Állapotonként (5% Tőkésítési Rátával)',
        xaxis_title='Műszaki Állapot',
        yaxis_title='Becsült Érték (Ft/m²)',
        template=PLOTLY_TEMPLATE,
        height=450
    )
    fig2.show()

# Kiadó lakások méret vs bérleti díj szórásdiagramja
fig3 = px.scatter(
    kiado,
    x='alapterulet_nm',
    y='price_huf',
    color='varosresz',
    trendline='ols',
    title='Alapterület vs. Havi bérleti díj a kiadó lakások piacán',
    labels={'alapterulet_nm': 'Alapterület (m²)', 'price_huf': 'Bérleti díj (HUF / hó)'},
    template=PLOTLY_TEMPLATE
)
fig3.update_layout(height=450)
fig3.show()

### 4. Interaktív Hozam- és Megtérülés Kalkulátor
Dinamikusan állítható kihasználtsági ráta, üzemeltetési költség és alapterület.

In [ ]:
w_occ = widgets.FloatSlider(min=0.7, max=1.0, step=0.05, value=0.95, description='Kihasználtság:')
w_cost = widgets.FloatSlider(min=0.05, max=0.30, step=0.05, value=0.15, description='Költség (%):')
w_ar_berlet = widgets.IntSlider(min=150, max=500, step=10, value=250, description='Bérlet (ezer):')
w_ar_vetel = widgets.IntSlider(min=30, max=120, step=5, value=50, description='Vételár (M):')

out_yield = widgets.Output()

def szamol_hozam(*args):
    eves_bev = (w_ar_berlet.value * 1e3 * 12) * w_occ.value
    netto_bev = eves_bev * (1.0 - w_cost.value)
    vetel = w_ar_vetel.value * 1e6
    brutto_h = (w_ar_berlet.value * 1e3 * 12 / vetel) * 100
    netto_h = (netto_bev / vetel) * 100
    megterules = vetel / netto_bev if netto_bev > 0 else 0
    
    with out_yield:
        clear_output(wait=True)
        res_kpis = [
            ("Bruttó Hozam", f"{brutto_h:.2f}%", "Éves bruttó", "#2563eb"),
            ("Nettó Hozam", f"{netto_h:.2f}%", f"Költség & üresedés után", "#10b981"),
            ("Nettó Éves Bevétel", fmt_huf(netto_bev), "Adózás előtti cash flow", "#059669"),
            ("Valós Megtérülés", f"{megterules:.1f} év", "Nettó cash flow alapján", "#d97706")
        ]
        display(HTML(kpi_grid_html(res_kpis)))

w_occ.observe(szamol_hozam, names='value')
w_cost.observe(szamol_hozam, names='value')
w_ar_berlet.observe(szamol_hozam, names='value')
w_ar_vetel.observe(szamol_hozam, names='value')

display(widgets.VBox([
    widgets.HBox([w_occ, w_cost]),
    widgets.HBox([w_ar_berlet, w_ar_vetel])
]))
display(out_yield)
szamol_hozam()